In [2]:
# ── 셀 0 │ 캐시 무효화 ──
import os
p = NEW + 'data/graphs/clinical_graphs.pt'
if os.path.exists(p):
    os.remove(p); print('삭제:', p)
else:
    print('캐시 없음 — 그대로 진행')

삭제: C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/graphs/clinical_graphs.pt


In [3]:
# ── 셀 1 │ 설정 ──
import os, sys, json, random, datetime, copy
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import add_self_loops, softmax
from sklearn.model_selection import StratifiedKFold
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.*')

NEW      = 'C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/'
FEAT_DIR = NEW + '01_pre-train/'
CLIN     = NEW + 'data/processed/clinical/clinical_pool_endpoints.csv'
GOUT     = NEW + 'data/graphs/'
CKPT     = NEW + 'models/pretrain/'
OUT      = NEW + 'results/finetune/'
os.makedirs(OUT, exist_ok=True)

if FEAT_DIR not in sys.path: sys.path.insert(0, FEAT_DIR)
import featurizer as FZ
ATOM_DIM, BOND_DIM = FZ.num_atom_features(), FZ.num_bond_features()
assert (ATOM_DIM, BOND_DIM) == (39, 10)

def _prep(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None or m.GetNumAtoms() == 0 or m.GetNumBonds() == 0: return None
    try: Chem.AssignStereochemistry(m, cleanIt=True, force=True)
    except Exception: pass
    return m

def make_graph(smi, y=None):
    mol = _prep(smi)
    if mol is None: return None
    try: x = np.stack([FZ.atom_features(a) for a in mol.GetAtoms()])
    except Exception: return None
    src, dst, ea = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = FZ.bond_features(b)
        src += [i, j]; dst += [j, i]; ea += [bf, bf]
    if not src: return None
    g = Data(x=torch.tensor(np.asarray(x, np.float32)),
             edge_index=torch.tensor([src, dst], dtype=torch.long),
             edge_attr=torch.tensor(np.asarray(ea, np.float32)))
    if y is not None: g.y = torch.tensor([float(y)], dtype=torch.float)
    return g

SEED = 42
HID, HEADS, DROP = 32, 8, 0.3
EPOCHS, PATIENCE = 60, 12
LR_BB, LR_CLF, WD = 1e-4, 5e-3, 5e-4
BS_TR, BS_EV = 32, 64
N_FOLD, N_REP = 5, 3          # A는 양성 45개라 뒤에서 반복을 늘립니다
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print('device', DEV)

device cuda


In [4]:
# ── 셀 2 │ 임상 그래프 생성 ──
def pick(df, *c, required=True):
    for x in c:
        if x in df.columns: return x
    if required: raise KeyError(f'{c} 중 없음: {list(df.columns)}')

clin = pd.read_csv(CLIN)
KEY = pick(clin, 'key14'); Y = pick(clin, 'y', 'Y', 'label')
SMI = pick(clin, 'canon_smiles', 'smiles', 'Standardized_SMILES')
print(f'임상 풀 {len(clin)}  양성 {int((clin[Y]==1).sum())}')

CPATH = GOUT + 'clinical_graphs.pt'
if os.path.exists(CPATH):
    blob = torch.load(CPATH, weights_only=False)
    cg, ck, crow = blob['graphs'], blob['keys'], blob['rows']
    print(f'캐시 로드 {len(cg)}개')
else:
    cg, ck, crow, fail = [], [], [], 0
    for i, r in clin.iterrows():
        g = make_graph(r[SMI], float(r[Y]))
        if g is None: fail += 1; continue
        cg.append(g); ck.append(r[KEY]); crow.append(i)
    torch.save({'graphs': cg, 'keys': ck, 'rows': crow}, CPATH)
    print(f'생성 {len(cg)} / 실패 {fail}')

C = clin.loc[crow].reset_index(drop=True)      # 그래프와 행 순서를 맞춘 표
assert len(C) == len(cg)

# 스캐폴드 (scaffold CV 용)
def scaffold(smi):
    try: return MurckoScaffold.MurckoScaffoldSmiles(smiles=str(smi), includeChirality=False)
    except Exception: return ''
C['scaffold'] = C[SMI].map(scaffold)
print(f'스캐폴드 {C["scaffold"].nunique()}종')

임상 풀 1225  양성 725
생성 1225 / 실패 0
스캐폴드 836종


In [5]:
# ── 셀 3 │ 엔드포인트 정의 ──
tox = C[Y] == 1
A = C['endpoint_A'] == 1 if 'endpoint_A' in C.columns else None
B = C['endpoint_B'] == 1 if 'endpoint_B' in C.columns else None
assert A is not None and B is not None, '셀 12(엔드포인트 확정)를 먼저 돌리세요'

ENDPOINTS = {
    'Broad': (tox,        ~tox),
    'A':     (A,          ~tox),
    'B':     (B,          ~tox),
    'C':     (A,          B   ),
}
for name, (p, n) in ENDPOINTS.items():
    print(f'{name:<6} 양성 {int(p.sum()):>4}  음성 {int(n.sum()):>4}  '
          f'합계 {int(p.sum()+n.sum()):>4}  양성률 {p.sum()/(p.sum()+n.sum()):.3f}')

def subset(ep):
    """엔드포인트별 (그래프 리스트, 라벨 배열, 스캐폴드 배열)."""
    p, n = ENDPOINTS[ep]
    m = (p | n).values
    idx = np.where(m)[0]
    lab = p.values[idx].astype(int)
    gs = []
    for i, lb in zip(idx, lab):
        g = copy.copy(cg[i])                    # 라벨만 엔드포인트에 맞게 교체
        g.y = torch.tensor([float(lb)], dtype=torch.float)
        gs.append(g)
    return gs, lab, C['scaffold'].values[idx]

Broad  양성  725  음성  500  합계 1225  양성률 0.592
A      양성  133  음성  500  합계  633  양성률 0.210
B      양성  592  음성  500  합계 1092  양성률 0.542
C      양성  133  음성  592  합계  725  양성률 0.183


In [6]:
# ── 셀 4 │ 모델 + loss + 평가 (사전학습 노트북과 동일) ──
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             accuracy_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix)

class GH(nn.Module):
    def __init__(s, ic, oc, d=DROP):
        super().__init__()
        s.W = nn.Parameter(torch.empty(ic, oc)); s.a = nn.Parameter(torch.empty(2*oc, 1))
        nn.init.xavier_uniform_(s.W); nn.init.xavier_uniform_(s.a)
        s.lr = nn.LeakyReLU(0.2); s.d = d
    def forward(s, x, ei):
        ei, _ = add_self_loops(ei, num_nodes=x.size(0))
        h = x @ s.W; row, col = ei
        e = s.lr((torch.cat([h[row], h[col]], -1) @ s.a).squeeze(-1))
        e = F.dropout(e, s.d, training=s.training)
        al = softmax(e, row)
        o = torch.zeros_like(h); o.index_add_(0, row, h[col] * al.unsqueeze(-1))
        return F.elu(o)

class GL(nn.Module):
    def __init__(s, ic, oc, h, d=DROP):
        super().__init__()
        s.h, s.d = h, d
        s.att = nn.ModuleList([GH(ic, oc, d) for _ in range(h)])
    def forward(s, x, ei):
        z = (torch.mean(torch.stack([a(x, ei) for a in s.att], 0), 0) if s.h == 1
             else torch.cat([a(x, ei) for a in s.att], -1))
        return F.dropout(z, s.d, training=s.training)

class AttentionPool(nn.Module):
    def __init__(s, ch, d=DROP):
        super().__init__()
        s.gate = nn.Sequential(nn.Linear(ch, ch), nn.ReLU(), nn.Dropout(d), nn.Linear(ch, 1))
    def forward(s, x, batch):
        al = softmax(s.gate(x).squeeze(-1), batch)
        out = torch.zeros(int(batch.max()) + 1, x.size(1), device=x.device, dtype=x.dtype)
        out.index_add_(0, batch, x * al.unsqueeze(-1))
        return out

class GAT_SI(nn.Module):
    """3층 GAT (8, 8, 1 heads) + attention pooling + 2층 FC classifier."""
    def __init__(s, ic, hc=HID, h=HEADS, d=DROP):
        super().__init__()
        s.l1 = GL(ic, hc, h, d)
        s.l2 = GL(hc*h, hc, h, d)
        s.l3 = GL(hc*h, hc, 1, d)
        s.pool = AttentionPool(hc, d)
        s.clf = nn.Sequential(nn.Linear(hc, 64), nn.ReLU(), nn.Dropout(d), nn.Linear(64, 1))
    def backbone_parameters(s):
        return (list(s.l1.parameters()) + list(s.l2.parameters())
                + list(s.l3.parameters()) + list(s.pool.parameters()))
    def forward(s, x, ei, b):
        x = F.elu(s.l1(x, ei)); x = F.elu(s.l2(x, ei)); x = s.l3(x, ei)
        return s.clf(s.pool(x, b)).squeeze(-1)

class FocalLoss(nn.Module):
    def __init__(s, alpha=0.25, gamma=2.0):
        super().__init__(); s.alpha, s.gamma = alpha, gamma
    def forward(s, logit, y):
        bce = F.binary_cross_entropy_with_logits(logit, y, reduction='none')
        p = torch.sigmoid(logit)
        pt = p*y + (1-p)*(1-y)
        at = s.alpha*y + (1-s.alpha)*(1-y)
        return (at * (1-pt)**s.gamma * bce).mean()

def make_loss(name, y_train):
    npos, nneg = max(int((y_train == 1).sum()), 1), max(int((y_train == 0).sum()), 1)
    if name == 'bce':   return nn.BCEWithLogitsLoss()
    if name == 'wbce':
        return nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([nneg/npos], dtype=torch.float, device=DEV))
    if name == 'focal': return FocalLoss(0.25, 2.0)
    raise ValueError(name)

def _at(ts, ps, th):
    pred = (ps >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(ts, pred, labels=[0, 1]).ravel()
    return {'threshold': float(th),
            'accuracy':  float(accuracy_score(ts, pred)),
            'bal_acc':   float(balanced_accuracy_score(ts, pred)),
            'f1':        float(f1_score(ts, pred, zero_division=0)),
            'precision': float(precision_score(ts, pred, zero_division=0)),
            'recall':    float(recall_score(ts, pred, zero_division=0)),
            'specificity': float(tn/(tn+fp)) if (tn+fp) else 0.0,
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def best_threshold(ts, ps):
    grid = np.arange(0.05, 0.96, 0.02)
    return float(max(grid, key=lambda t: f1_score(ts, ps >= t, zero_division=0)))

@torch.no_grad()
def evaluate(model, ld, fixed_thr=None, full=True):
    model.eval(); ps, ts = [], []
    for b in ld:
        b = b.to(DEV)
        ps += torch.sigmoid(model(b.x, b.edge_index, b.batch)).cpu().tolist()
        ts += b.y.view(-1).cpu().tolist()
    ps, ts = np.array(ps), np.array(ts, dtype=int)
    try: auroc = float(roc_auc_score(ts, ps))
    except ValueError: auroc = 0.5
    out = {'auroc': auroc, 'aucpr': float(average_precision_score(ts, ps)),
           'n': int(len(ts)), 'n_pos': int(ts.sum())}
    if full:
        out['at_050']   = _at(ts, ps, 0.5)
        out['at_f1opt'] = _at(ts, ps, best_threshold(ts, ps))
        if fixed_thr is not None:
            out['at_valthr'] = _at(ts, ps, fixed_thr)
    return out

print('모델·loss 준비 완료. 파라미터',
      f'{sum(p.numel() for p in GAT_SI(ATOM_DIM).parameters()):,}')

모델·loss 준비 완료. 파라미터 88,066


In [7]:
# ── 셀 5 │ CV 분할 + 학습 ──
from sklearn.model_selection import StratifiedKFold
from scipy import stats

LOSS = 'wbce'          # A는 양성률 16%라 가중을 씁니다

def strat_folds(lab, n_fold, seed):
    skf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(len(lab)), lab))

def scaffold_folds(lab, scaf, n_fold, seed):
    """스캐폴드 그룹을 통째로 한 fold에만 배정. 큰 그룹부터 가장 작은 fold에."""
    rng = np.random.RandomState(seed)
    groups = {}
    for i, s in enumerate(scaf):
        groups.setdefault(s if s else f'_empty{i}', []).append(i)
    order = sorted(groups.values(), key=lambda g: (-len(g), rng.rand()))
    bins = [[] for _ in range(n_fold)]
    for g in order:
        bins[int(np.argmin([len(b) for b in bins]))].extend(g)
    out = []
    for k in range(n_fold):
        te = np.array(sorted(bins[k]))
        tr = np.array(sorted(set(range(len(lab))) - set(te)))
        out.append((tr, te))
    return out

def load_backbone(model, ckpt_path):
    """사전학습 가중치에서 backbone(l1,l2,l3,pool)만 가져오고 분류기는 새로."""
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    keep = {k: v for k, v in sd.items() if not k.startswith('clf.')}
    missing, unexpected = model.load_state_dict(keep, strict=False)
    assert all(k.startswith('clf.') for k in missing), f'예상 밖 missing: {missing}'
    assert not unexpected, f'unexpected: {unexpected}'
    return model

def fit_fold(gs_tr, lab_tr, gs_va, lab_va, arm, ckpt, seed):
    seed_all(seed)
    model = GAT_SI(ATOM_DIM).to(DEV)
    if arm == 'transfer':
        model = load_backbone(model, ckpt).to(DEV)
    crit = make_loss(LOSS, np.asarray(lab_tr))
    opt = torch.optim.Adam(
        [{'params': model.backbone_parameters(), 'lr': LR_BB},
         {'params': model.clf.parameters(),      'lr': LR_CLF}], weight_decay=WD)

    L_tr = DataLoader(gs_tr, batch_size=BS_TR, shuffle=True)
    L_va = DataLoader(gs_va, batch_size=BS_EV, shuffle=False)
    best, best_state, wait = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for b in L_tr:
            b = b.to(DEV); opt.zero_grad()
            crit(model(b.x, b.edge_index, b.batch), b.y.view(-1).float()).backward()
            opt.step()
        v = evaluate(model, L_va, full=False)
        if v['auroc'] > best:
            best, wait = v['auroc'], 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
        if wait >= PATIENCE: break
    model.load_state_dict({k: t.to(DEV) for k, t in best_state.items()})
    return model, best

def run_cv(ep_name, split_type, arm, ckpt, n_rep=N_REP, n_fold=N_FOLD, verbose=True):
    gs, lab, scaf = subset(ep_name)
    rows = []
    for rep in range(n_rep):
        seed = SEED + rep
        folds = (strat_folds(lab, n_fold, seed) if split_type == 'stratified'
                 else scaffold_folds(lab, scaf, n_fold, seed))
        for k, (tr, te) in enumerate(folds):
            # 학습 fold 안에서 검증셋을 다시 떼어 early stopping
            inner = strat_folds(lab[tr], max(n_fold, 5), seed)[0]
            tr2, va = tr[inner[0]], tr[inner[1]]
            model, bv = fit_fold([gs[i] for i in tr2], lab[tr2],
                                 [gs[i] for i in va],  lab[va], arm, ckpt, seed * 100 + k)
            r = evaluate(model, DataLoader([gs[i] for i in te],
                                           batch_size=BS_EV, shuffle=False))
            a = r['at_050']
            rows.append({'endpoint': ep_name, 'split': split_type, 'arm': arm,
                         'rep': rep, 'fold': k, 'n_test': r['n'], 'n_pos': r['n_pos'],
                         'auroc': r['auroc'], 'aucpr': r['aucpr'],
                         'accuracy': a['accuracy'], 'f1': a['f1'],
                         'precision': a['precision'], 'recall': a['recall'],
                         'val_auroc': bv})
            if verbose:
                print(f'  {ep_name:<6} {split_type:<10} {arm:<8} rep{rep} fold{k}  '
                      f'AUROC {r["auroc"]:.4f}  AUCPR {r["aucpr"]:.4f}')
    return pd.DataFrame(rows)

In [8]:
# ── 셀 6 │ 실행 ──
CKPT_FILE = CKPT + f'pretrain_herg_karim_{LOSS}.pt'
assert os.path.exists(CKPT_FILE), CKPT_FILE
print('사전학습 체크포인트:', CKPT_FILE, '\n')

all_rows = []
for ep_name in ['Broad', 'A', 'B', 'C']:
    for split_type in ['stratified', 'scaffold']:
        for arm in ['transfer', 'scratch']:
            t0 = datetime.datetime.now()
            df = run_cv(ep_name, split_type, arm, CKPT_FILE, verbose=False)
            all_rows.append(df)
            mins = (datetime.datetime.now() - t0).total_seconds() / 60
            print(f'{ep_name:<6} {split_type:<10} {arm:<8}  '
                  f'AUROC {df["auroc"].mean():.4f} ± {df["auroc"].std():.4f}  '
                  f'AUCPR {df["aucpr"].mean():.4f}  ({mins:.1f}분)')

R = pd.concat(all_rows, ignore_index=True)
R.to_csv(OUT + 'cv_folds_raw.csv', index=False, encoding='utf-8-sig')
print('\n저장:', OUT + 'cv_folds_raw.csv')

사전학습 체크포인트: C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/models/pretrain/pretrain_herg_karim_wbce.pt 

Broad  stratified transfer  AUROC 0.6565 ± 0.0347  AUCPR 0.7201  (6.5분)
Broad  stratified scratch   AUROC 0.6510 ± 0.0275  AUCPR 0.7309  (8.0분)
Broad  scaffold   transfer  AUROC 0.6528 ± 0.0281  AUCPR 0.7245  (9.0분)
Broad  scaffold   scratch   AUROC 0.6402 ± 0.0447  AUCPR 0.7314  (7.8분)
A      stratified transfer  AUROC 0.8366 ± 0.0358  AUCPR 0.5633  (4.3분)
A      stratified scratch   AUROC 0.8208 ± 0.0470  AUCPR 0.5686  (4.5분)
A      scaffold   transfer  AUROC 0.8155 ± 0.0605  AUCPR 0.5299  (4.1분)
A      scaffold   scratch   AUROC 0.8148 ± 0.0518  AUCPR 0.5429  (5.4분)
B      stratified transfer  AUROC 0.6225 ± 0.0453  AUCPR 0.6459  (6.9분)
B      stratified scratch   AUROC 0.6068 ± 0.0488  AUCPR 0.6409  (7.8분)
B      scaffold   transfer  AUROC 0.6075 ± 0.0475  AUCPR 0.6367  (6.9분)
B      scaffold   scratch   AUROC 0.5960 ± 0.0265  AUCPR 0.6344  (7.9분)
C      stra

In [9]:
# ── 셀 7 │ 집계 + transfer vs clinical-only 검정 ──
from scipy import stats

R = pd.read_csv(OUT + 'cv_folds_raw.csv')     # 커널 새로 열었으면 이 줄부터
METRICS = ['accuracy', 'auroc', 'aucpr', 'f1', 'precision', 'recall']
NAME = {'scratch': 'Clinical only', 'transfer': 'Transfer learned'}

# 1) clinical-only 단독 보기
print('='*78)
print('[Clinical only — 엔드포인트 × split]')
co = (R[R['arm'] == 'scratch']
      .groupby(['endpoint', 'split'])[METRICS]
      .agg(['mean', 'std']).round(4))
print(co.to_string())

# 2) 평균 ± 표준편차 (논문 표 형식)
def fmt(df):
    return df.groupby(['endpoint', 'split', 'arm'])[METRICS].apply(
        lambda g: pd.Series({m: f'{g[m].mean():.4f}±{g[m].std():.4f}' for m in METRICS}))

print('\n' + '='*78)
print('[전체 — mean ± std over 15 folds]')
print(fmt(R).to_string())

# 3) fold 쌍을 맞춘 검정
print('\n' + '='*78)
print('[Transfer − Clinical only, paired over folds]')
rows = []
for (ep, sp), g in R.groupby(['endpoint', 'split']):
    t = g[g['arm'] == 'transfer'].sort_values(['rep', 'fold'])
    s = g[g['arm'] == 'scratch' ].sort_values(['rep', 'fold'])
    assert (t[['rep','fold']].values == s[['rep','fold']].values).all(), 'fold 불일치'
    r = {'endpoint': ep, 'split': sp, 'n_fold': len(t)}
    for m in ['auroc', 'aucpr', 'f1']:
        a, b = t[m].values, s[m].values
        d = a - b
        try:    _, p = stats.wilcoxon(a, b)
        except Exception: p = np.nan
        _, pt = stats.ttest_rel(a, b)
        r[f'{m}_transfer'] = a.mean()
        r[f'{m}_clinical'] = b.mean()
        r[f'{m}_delta']    = d.mean()
        r[f'{m}_p_wilcox'] = p
        r[f'{m}_p_ttest']  = pt
        # 95% CI (fold 차이의 t 분포 기준)
        se = d.std(ddof=1) / np.sqrt(len(d))
        r[f'{m}_ci_lo'] = d.mean() - 1.96*se
        r[f'{m}_ci_hi'] = d.mean() + 1.96*se
    rows.append(r)

P = pd.DataFrame(rows)
show = ['endpoint', 'split', 'n_fold',
        'auroc_clinical', 'auroc_transfer', 'auroc_delta',
        'auroc_ci_lo', 'auroc_ci_hi', 'auroc_p_wilcox']
print(P[show].round(4).to_string(index=False))

P.to_csv(OUT + 'transfer_vs_clinical.csv', index=False, encoding='utf-8-sig')

[Clinical only — 엔드포인트 × split]
                    accuracy           auroc           aucpr              f1         precision          recall        
                        mean     std    mean     std    mean     std    mean     std      mean     std    mean     std
endpoint split                                                                                                        
A        scaffold     0.7224  0.0548  0.8148  0.0518  0.5429  0.1005  0.5344  0.0711    0.4181  0.0733  0.7633  0.1232
         stratified   0.7256  0.0682  0.8208  0.0470  0.5686  0.1009  0.5474  0.0719    0.4310  0.0850  0.7748  0.1036
B        scaffold     0.5632  0.0281  0.5960  0.0265  0.6344  0.0476  0.5482  0.1357    0.6120  0.0457  0.5278  0.1696
         stratified   0.5726  0.0333  0.6068  0.0488  0.6409  0.0489  0.5957  0.0703    0.6146  0.0412  0.6026  0.1590
Broad    scaffold     0.5897  0.0487  0.6402  0.0447  0.7314  0.0373  0.6020  0.0756    0.7048  0.0625  0.5364  0.1062
         stratif

In [10]:
# ── 셀 8 │ 원고 표 형식 (Clinical only / Transfer learned 행) ──
COLS = ['Accuracy', 'AUCROC', 'AUCPR', 'F1 score', 'Precision', 'Recall']
MAP = dict(zip(COLS, METRICS))

def paper_table(ep, sp):
    out = {}
    for arm in ['scratch', 'transfer']:
        g = R[(R['endpoint'] == ep) & (R['split'] == sp) & (R['arm'] == arm)]
        out[NAME[arm]] = {c: f'{g[MAP[c]].mean():.4f}±{g[MAP[c]].std():.4f}' for c in COLS}
    return pd.DataFrame(out).T[COLS]

for ep in ['Broad', 'A', 'B', 'C']:
    for sp in ['stratified', 'scaffold']:
        if not ((R['endpoint'] == ep) & (R['split'] == sp)).any():
            continue
        print(f'\n[{ep} / {sp}]')
        t = paper_table(ep, sp)
        print(t.to_string())
        print('  (탭 구분)')
        print('\t' + '\t'.join(COLS))
        for i, r in t.iterrows():
            print(i + '\t' + '\t'.join(r[c] for c in COLS))


[Broad / stratified]
                       Accuracy         AUCROC          AUCPR       F1 score      Precision         Recall
Clinical only     0.6114±0.0287  0.6510±0.0275  0.7309±0.0357  0.6482±0.0616  0.6996±0.0460  0.6244±0.1460
Transfer learned  0.6128±0.0236  0.6565±0.0347  0.7201±0.0461  0.6582±0.0355  0.6899±0.0299  0.6354±0.0778
  (탭 구분)
	Accuracy	AUCROC	AUCPR	F1 score	Precision	Recall
Clinical only	0.6114±0.0287	0.6510±0.0275	0.7309±0.0357	0.6482±0.0616	0.6996±0.0460	0.6244±0.1460
Transfer learned	0.6128±0.0236	0.6565±0.0347	0.7201±0.0461	0.6582±0.0355	0.6899±0.0299	0.6354±0.0778

[Broad / scaffold]
                       Accuracy         AUCROC          AUCPR       F1 score      Precision         Recall
Clinical only     0.5897±0.0487  0.6402±0.0447  0.7314±0.0373  0.6020±0.0756  0.7048±0.0625  0.5364±0.1062
Transfer learned  0.6125±0.0331  0.6528±0.0281  0.7245±0.0253  0.6466±0.0540  0.6981±0.0298  0.6074±0.0846
  (탭 구분)
	Accuracy	AUCROC	AUCPR	F1 score	Precision	Recall
C

In [11]:
# ── 셀 11 │ 전체 임상 풀로 최종 모델 학습 후 저장 ──
FT = NEW + 'models/finetune/'
os.makedirs(FT, exist_ok=True)

gs_all, lab_all, _ = subset('Broad')
N_SEED = 3

@torch.no_grad()
def probs(model, ld):
    model.eval(); out = []
    for b in ld:
        b = b.to(DEV)
        out += torch.sigmoid(model(b.x, b.edge_index, b.batch)).cpu().tolist()
    return np.array(out)

manifest = []
for arm in ['scratch', 'transfer']:
    for s in range(N_SEED):
        seed = SEED + s
        tr, va = strat_folds(lab_all, 10, seed)[0]     # 10% 내부 검증
        model, bv = fit_fold([gs_all[i] for i in tr], lab_all[tr],
                             [gs_all[i] for i in va], lab_all[va],
                             arm, CKPT_FILE, seed)

        L_va = DataLoader([gs_all[i] for i in va], batch_size=BS_EV, shuffle=False)
        pv, yv = probs(model, L_va), lab_all[va]
        thr = best_threshold(yv, pv)

        name = f'finetune_{arm}_seed{seed}.pt'
        torch.save({'state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
                    'arm': arm, 'seed': seed, 'loss': LOSS,
                    'atom_dim': ATOM_DIM, 'hidden': HID, 'heads': HEADS, 'dropout': DROP,
                    'val_idx': va.tolist(), 'val_threshold': float(thr),
                    'val_auroc': float(bv),
                    'pretrain_ckpt': CKPT_FILE if arm == 'transfer' else None},
                   FT + name)
        manifest.append({'file': name, 'arm': arm, 'seed': seed,
                         'val_threshold': round(thr, 3), 'val_auroc': round(bv, 4)})
        print(f'  저장 {name}  thr {thr:.2f}  val AUROC {bv:.4f}')

json.dump(manifest, open(FT + 'manifest.json', 'w', encoding='utf-8'),
          indent=2, ensure_ascii=False)
print('\n', pd.DataFrame(manifest).to_string(index=False))

  저장 finetune_scratch_seed42.pt  thr 0.33  val AUROC 0.6518
  저장 finetune_scratch_seed43.pt  thr 0.33  val AUROC 0.6452
  저장 finetune_scratch_seed44.pt  thr 0.43  val AUROC 0.7362
  저장 finetune_transfer_seed42.pt  thr 0.39  val AUROC 0.7164
  저장 finetune_transfer_seed43.pt  thr 0.23  val AUROC 0.6537
  저장 finetune_transfer_seed44.pt  thr 0.45  val AUROC 0.7490

                        file      arm  seed  val_threshold  val_auroc
 finetune_scratch_seed42.pt  scratch    42           0.33     0.6518
 finetune_scratch_seed43.pt  scratch    43           0.33     0.6452
 finetune_scratch_seed44.pt  scratch    44           0.43     0.7362
finetune_transfer_seed42.pt transfer    42           0.39     0.7164
finetune_transfer_seed43.pt transfer    43           0.23     0.6537
finetune_transfer_seed44.pt transfer    44           0.45     0.7490


In [12]:
# ── 셀 9 │ 논문 표 레이아웃 (split × endpoint) ──
EP_LABEL = {'Broad': 'Overall', 'A': 'A(listed)',
            'B': "B'(non-listed)", 'C': "C'(listed vs non-listed)"}
SP_LABEL = {'stratified': 'Stratified', 'scaffold': 'Scaffold'}
COLS   = ['Accuracy', 'AUCROC', 'AUCPR', 'f1 score', 'precision', 'recall']
MCOL   = dict(zip(COLS, ['accuracy', 'auroc', 'aucpr', 'f1', 'precision', 'recall']))

def layout(arm):
    rows, idx = [], []
    for sp in ['stratified', 'scaffold']:
        for ep in ['Broad', 'A', 'B', 'C']:
            g = R[(R['arm'] == arm) & (R['split'] == sp) & (R['endpoint'] == ep)]
            if g.empty: continue
            rows.append({c: f'{g[MCOL[c]].mean():.4f} ± {g[MCOL[c]].std():.4f}'
                         for c in COLS})
            idx.append((SP_LABEL[sp], EP_LABEL[ep]))
    return pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(
        idx, names=['CV', 'Endpoint']))[COLS]

for arm, title in [('transfer', 'Transfer learned'), ('scratch', 'Clinical only')]:
    t = layout(arm)
    print('=' * 100)
    print(f'[{title}]')
    print(t.to_string())
    t.to_csv(OUT + f'table_layout_{arm}.csv', encoding='utf-8-sig')

    print(f'\n  (탭 구분 — {title})')
    print('CV\tEndpoint\t' + '\t'.join(COLS))
    prev = None
    for (sp, ep), r in t.iterrows():
        print(('' if sp == prev else sp) + '\t' + ep + '\t'
              + '\t'.join(r[c] for c in COLS))
        prev = sp
    print()

[Transfer learned]
                                        Accuracy           AUCROC            AUCPR         f1 score        precision           recall
CV         Endpoint                                                                                                                  
Stratified Overall               0.6128 ± 0.0236  0.6565 ± 0.0347  0.7201 ± 0.0461  0.6582 ± 0.0355  0.6899 ± 0.0299  0.6354 ± 0.0778
           A(QT/TdP)             0.7573 ± 0.0472  0.8366 ± 0.0358  0.5633 ± 0.0669  0.5692 ± 0.0625  0.4581 ± 0.0579  0.7602 ± 0.0964
           B'(non-QT)            0.5876 ± 0.0380  0.6225 ± 0.0453  0.6459 ± 0.0479  0.6058 ± 0.0629  0.6250 ± 0.0279  0.5941 ± 0.0991
           C'(QT/TdP vs non-QT)  0.6359 ± 0.0459  0.7225 ± 0.0524  0.3694 ± 0.0688  0.4198 ± 0.0490  0.2978 ± 0.0367  0.7176 ± 0.1037
Scaffold   Overall               0.6125 ± 0.0331  0.6528 ± 0.0281  0.7245 ± 0.0253  0.6466 ± 0.0540  0.6981 ± 0.0298  0.6074 ± 0.0846
           A(QT/TdP)             0.7582 ± 0

In [13]:
# ── 셀 9 │ 논문 표 레이아웃 (split × endpoint) ──
EP_LABEL = {'Broad': 'Overall', 'A': 'A(listed)',
            'B': "B'(non-listed)", 'C': "C'(listed vs non-listed)"}
SP_LABEL = {'stratified': 'Stratified', 'scaffold': 'Scaffold'}
COLS   = ['Accuracy', 'AUCROC', 'AUCPR', 'f1 score', 'precision', 'recall']
MCOL   = dict(zip(COLS, ['accuracy', 'auroc', 'aucpr', 'f1', 'precision', 'recall']))

def layout(arm):
    rows, idx = [], []
    for sp in ['stratified', 'scaffold']:
        for ep in ['Broad', 'A', 'B', 'C']:
            g = R[(R['arm'] == arm) & (R['split'] == sp) & (R['endpoint'] == ep)]
            if g.empty: continue
            rows.append({c: f'{g[MCOL[c]].mean():.4f} ± {g[MCOL[c]].std():.4f}'
                         for c in COLS})
            idx.append((SP_LABEL[sp], EP_LABEL[ep]))
    return pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(
        idx, names=['CV', 'Endpoint']))[COLS]

for arm, title in [('transfer', 'Transfer learned'), ('scratch', 'Clinical only')]:
    t = layout(arm)
    print('=' * 100)
    print(f'[{title}]')
    print(t.to_string())
    t.to_csv(OUT + f'table_layout_{arm}.csv', encoding='utf-8-sig')

    print(f'\n  (탭 구분 — {title})')
    print('CV\tEndpoint\t' + '\t'.join(COLS))
    prev = None
    for (sp, ep), r in t.iterrows():
        print(('' if sp == prev else sp) + '\t' + ep + '\t'
              + '\t'.join(r[c] for c in COLS))
        prev = sp
    print()

[Transfer learned]
                                            Accuracy           AUCROC            AUCPR         f1 score        precision           recall
CV         Endpoint                                                                                                                      
Stratified Overall                   0.6128 ± 0.0236  0.6565 ± 0.0347  0.7201 ± 0.0461  0.6582 ± 0.0355  0.6899 ± 0.0299  0.6354 ± 0.0778
           A(listed)                 0.7573 ± 0.0472  0.8366 ± 0.0358  0.5633 ± 0.0669  0.5692 ± 0.0625  0.4581 ± 0.0579  0.7602 ± 0.0964
           B'(non-listed)            0.5876 ± 0.0380  0.6225 ± 0.0453  0.6459 ± 0.0479  0.6058 ± 0.0629  0.6250 ± 0.0279  0.5941 ± 0.0991
           C'(listed vs non-listed)  0.6359 ± 0.0459  0.7225 ± 0.0524  0.3694 ± 0.0688  0.4198 ± 0.0490  0.2978 ± 0.0367  0.7176 ± 0.1037
Scaffold   Overall                   0.6125 ± 0.0331  0.6528 ± 0.0281  0.7245 ± 0.0253  0.6466 ± 0.0540  0.6981 ± 0.0298  0.6074 ± 0.0846
           A(li